# Naive Bayes Híbrido — Dados Contínuos e Discretos

Este notebook implementa um classificador **Naive Bayes híbrido**:

- **Gaussiano** → atributos contínuos (usa média e desvio padrão)
- **Categórico** → atributos discretos (usa frequência relativa)

### Dataset: `load_wine` (sklearn)
178 amostras de vinhos classificados em 3 tipos (class_0, class_1, class_2).

| Coluna | Tipo real |
|---|---|
| alcohol, malic_acid, ash, alcalinity_of_ash, total_phenols, flavanoids, nonflavanoid_phenols, proanthocyanins, color_intensity, hue, od280/od315 | **Contínuo** |
| magnesium | **Discreto** |
| proline | **Discreto** |

## 1. Importações

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 2. Carregamento do Dataset

In [2]:
data = load_wine()
X = data.data
y = data.target

print("Atributos:", data.feature_names)
print(f"\nShape: {X.shape}")
print(f"Classes: {data.target_names}")

idx_continuo = [0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11]
idx_discreto = [4, 12]

Atributos: ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']

Shape: (178, 13)
Classes: ['class_0' 'class_1' 'class_2']


## 3. Implementação do Naive Bayes Híbrido

A regra de decisão é o **Teorema de Bayes**:

$$P(C \mid x) \propto P(C) \cdot \prod_{j} P(x_j \mid C)$$

Para atributos **contínuos**, estimamos $P(x_j \mid C)$ pela densidade Gaussiana:

$$P(x_j \mid C) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(x_j - \mu)^2}{2\sigma^2}\right)$$

Para atributos **discretos**, estimamos $P(x_j \mid C)$ pela frequência relativa:

$$P(x_j \mid C) = \frac{\sum_{i=1}^{n} (x_j, C)}{n_C}$$

In [3]:
class naiveBayes:
    """
    Naive Bayes Híbrido.
    Adaptado do modelo base do professor para suportar atributos
    contínuos (Gaussiano) e discretos (frequência relativa).
    """

    def __init__(self, idx_continuo, idx_discreto):
        """
        Parâmetros
        ----------
        idx_continuo : list — índices das colunas contínuas
        idx_discreto : list — índices das colunas discretas
        """
        self.idx_continuo  = idx_continuo
        self.idx_discreto  = idx_discreto
        self.classes       = None
        self.prob_classe   = []  # P(C) — probabilidade a priori
        self.media         = []  # média por classe (colunas contínuas)
        self.std           = []  # desvio padrão por classe (colunas contínuas)
        self.freq_discreta = []  # tabela de frequência por classe (colunas discretas)
        self.valores_disc  = []  # valores únicos por coluna discreta

    # ------------------------------------------------------------------
    def fit(self, X, y):
        self.classes = np.unique(y)

        # Valores únicos de cada coluna discreta
        self.valores_disc = [np.unique(X[:, j]) for j in self.idx_discreto]

        for c in self.classes:
            dados_c = X[y == c]

            # P(C)
            self.prob_classe.append(len(dados_c) / len(X))

            # Parâmetros gaussianos (apenas colunas contínuas)
            self.media.append(np.mean(dados_c[:, self.idx_continuo], axis=0))
            self.std.append(np.std(dados_c[:, self.idx_continuo], axis=0))

            # Frequências relativas (apenas colunas discretas)
            freq_c = {}
            for k, j in enumerate(self.idx_discreto):
                col = dados_c[:, j]
                n_c = len(col)
                freq_c[j] = {
                    val: np.sum(col == val) / n_c
                    for val in self.valores_disc[k]
                }
            self.freq_discreta.append(freq_c)

    # ------------------------------------------------------------------
    def gauss(self, media, std, x):
        """Densidade Gaussiana"""
        return 1 / np.sqrt(2 * np.pi * std**2) * np.e ** (-((x - media)**2) / (2 * std**2))

    # ------------------------------------------------------------------
    def predict(self, X):
        preds = []
        for x in X:
            log_probs = []
            for i in range(len(self.classes)):

                # log P(C)
                log_p = np.log(self.prob_classe[i])

                # log P(x_cont | C) — Gaussiano
                pg = self.gauss(self.media[i], self.std[i], x[self.idx_continuo])
                pg = np.clip(pg, 1e-300, None)  # evita log(0)
                log_p += np.sum(np.log(pg))

                # log P(x_disc | C) — Frequência relativa
                for k, j in enumerate(self.idx_discreto):
                    val = x[j]
                    p_d = self.freq_discreta[i][j].get(val, 0)
                    p_d = np.clip(p_d, 1e-300, None) # evita log(0)
                    log_p += np.log(p_d)

                log_probs.append(log_p)

            preds.append(self.classes[np.argmax(log_probs)])
        return np.array(preds)

## 4. Treinamento e Avaliação

In [4]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Treino : {X_treino.shape[0]} amostras")
print(f"Teste  : {X_teste.shape[0]} amostras")

Treino : 124 amostras
Teste  : 54 amostras


In [5]:
nb = naiveBayes(idx_continuo=idx_continuo, idx_discreto=idx_discreto)
nb.fit(X_treino, y_treino)
predicoes = nb.predict(X_teste)

acc = accuracy_score(y_teste, predicoes)
print(f"Acurácia no conjunto de teste: {acc * 100:.2f}%")

Acurácia no conjunto de teste: 83.33%


## 5. Matriz de Confusão

In [6]:
cm = confusion_matrix(y_teste, predicoes)
print("Matriz de Confusão:")
print(cm)

Matriz de Confusão:
[[16  0  2]
 [ 1 19  1]
 [ 2  3 10]]


## 6. Predição de uma Nova Amostra

In [7]:
# Usando a primeira amostra do conjunto de teste como exemplo
nova_amostra = X_teste[0].reshape(1, -1)
resultado    = nb.predict(nova_amostra)[0]
real         = y_teste[0]

print(f"Predição : {data.target_names[resultado]}")
print(f"Real     : {data.target_names[real]}")
print(f"Correto  : {resultado == real}")

Predição : class_0
Real     : class_0
Correto  : True
